## Basic import


In [ ]:
# 修复：增加了 langchain-chroma
!pip install -qU langchain langchain-openai langchain-community langchain-chroma chromadb ragas datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72

In [ ]:
!pip install langchain-chroma langchain-huggingface sentence-transformers datasets tqdm

In [ ]:
import os
import getpass

# 请在这里输入你的 OpenAI API Key
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

Enter your OpenAI API Key: ··········


In [ ]:
import torch
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings # 替换了 OpenAIEmbeddings
from tqdm import tqdm

# ================= 配置区 =================
SAMPLE_SIZE = 5000   # 5000条数据 ≈ 50,000 个文档片段 (模拟大海捞针)
BATCH_SIZE = 64      # HuggingFace 本地推理显存占用较大，建议 Batch 设小一点 (64 或 32)
# 推荐模型：
# 1. BAAI/bge-m3 (多语言，最强，显存要求稍高)
# 2. BAAI/bge-small-en-v1.5 (纯英文，速度极快，省显存)
MODEL_NAME = "BAAI/bge-small-en-v1.5"
# =========================================

# 0. 检查设备 (自动选择 GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"正在使用设备: {device.upper()}")
if device == "cpu":
    print("⚠️ 警告：检测到正在使用 CPU。构建 5000 条数据的索引会非常慢！建议在 Colab 开启 GPU Runtime。")

# 1. 加载数据
print(f"正在加载数据集 (前 {SAMPLE_SIZE} 条)...")
dataset = load_dataset("hotpot_qa", "fullwiki", split=f"train[:{SAMPLE_SIZE}]")

# 2. 数据预处理
print("正在构建文档库...")
docs = []
for row in tqdm(dataset, desc="Processing Contexts"):
    titles = row['context']['title']
    sentences = row['context']['sentences']

    for title, sent_list in zip(titles, sentences):
        text_content = f"{title}\n{' '.join(sent_list)}"
        docs.append(Document(
            page_content=text_content,
            metadata={"source": title, "related_question_id": row['id']}
        ))

print(f"预处理完成。共生成 {len(docs)} 个文档片段。")

# 3. 初始化 HuggingFace Embeddings (关键修改点)
print(f"正在加载模型 {MODEL_NAME} 到 {device}...")
embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs={'device': device},      # 强制使用 GPU
    encode_kwargs={'normalize_embeddings': True} # 归一化，这对余弦相似度检索很重要
)

# 4. 构建向量数据库
# 注意：第一次运行需要下载模型权重，可能需要几分钟
print("开始构建向量索引 (这可能需要几分钟)...")

vectorstore = Chroma(
    collection_name="hotpot_qa_hf_bigsea",
    embedding_function=embeddings,
    persist_directory="./chroma_db_hf_bigsea"
)

# 分批写入 (防止爆显存)
total_docs = len(docs)
for i in tqdm(range(0, total_docs, BATCH_SIZE), desc="Indexing Vectors"):
    batch = docs[i : i + BATCH_SIZE]
    vectorstore.add_documents(batch)

print(f"\n✅ 向量数据库构建完成！")
print(f"当前环境：模拟 Full Wiki 检索 (Local Embedding)")
print(f"文档总量：{total_docs} 个片段")

# ================= 测试代码 =================
# 随机取一条测试
test_idx = 0
question = dataset[test_idx]['question']
ground_truth_titles = [fact[0] for fact in dataset[test_idx]['supporting_facts']]

print(f"\n--- 测试问题 ---")
print(f"Question: {question}")
print(f"Looking for answers in: {ground_truth_titles}")

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
retrieved_docs = retriever.invoke(question)

print(f"\n--- 检索结果 ---")
for i, doc in enumerate(retrieved_docs):
    # 检查是否命中
    is_hit = "✅ HIT!" if doc.metadata['source'] in ground_truth_titles else "❌"
    print(f"[{i+1}] {is_hit} {doc.metadata['source']}")

正在使用设备: CUDA
正在加载数据集 (前 5000 条)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

fullwiki/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

开始构建向量索引 (这可能需要几分钟)...


Indexing Vectors: 100%|██████████| 777/777 [05:49<00:00,  2.23it/s]


✅ 向量数据库构建完成！
当前环境：模拟 Full Wiki 检索 (Local Embedding)
文档总量：49708 个片段

--- 测试问题 ---
Question: Which magazine was started first Arthur's Magazine or First for Women?
Looking for answers in: ['t', 's']

--- 检索结果 ---
[1] ❌ Arthur's Magazine
[2] ❌ First for Women
[3] ❌ First for Women
[4] ❌ Arthur's Lady's Home Magazine
[5] ❌ Peterson's Magazine


## Raw RAG

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. 定义 LLM
llm = ChatOpenAI(model="gpt-5-mini")

# 2. 定义 Prompt 模板
template = """You are a helpful assistant. Answer the question based only on the following context.

Context:
{context}

Question:
{question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# 3. 辅助函数：格式化检索到的文档
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

# 4. 构建 Chain (User Query -> Embed Query -> Vector Search -> Top-K -> LLM -> Response)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG Chain 已就绪。")

RAG Chain 已就绪。


In [ ]:
from datasets import Dataset
from tqdm import tqdm

# ================= 修正版推理代码 (带实时打印) =================

# 1. 决定测试集大小
EVAL_SIZE = 50

# 从 5000 条的全量 dataset 中，切片前 50 条作为“考卷”
test_dataset = dataset.select(range(EVAL_SIZE))

print(f"开始批量推理 (仅评估前 {EVAL_SIZE} 条)...")
print("-" * 50) # 打印分割线

ragas_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

# 遍历测试集
for i, row in enumerate(tqdm(test_dataset)):
    question = row['question']
    ground_truth = row['answer']

    # 1. 检索 (去 5000 条的大库里搜)
    retrieved_docs = retriever.invoke(question)
    contexts = [doc.page_content for doc in retrieved_docs]

    # 2. 生成 (模型回答)
    answer = rag_chain.invoke(question)

    # ================= 实时打印区 (新加的) =================
    print(f"\n[第 {i+1} 题]")
    print(f"🔍 Question: {question}")
    print(f"🤖 Model Answer: {answer}")
    print(f"📚 Ground Truth: {ground_truth}")
    print("-" * 30)
    # ====================================================

    # 3. 收集数据 (用于后续 RAGAS 评分)
    ragas_data["question"].append(question)
    ragas_data["answer"].append(answer)
    ragas_data["contexts"].append(contexts)
    ragas_data["ground_truth"].append(ground_truth)

# 转换格式给 RAGAS 使用
eval_dataset = Dataset.from_dict(ragas_data)
print("\n✅ 评估数据准备完毕，eval_dataset 已生成。")

开始批量推理 (仅评估前 50 条)...
--------------------------------------------------


  2%|▏         | 1/50 [00:04<04:04,  4.99s/it]


[第 1 题]
🔍 Question: Which magazine was started first Arthur's Magazine or First for Women?
🤖 Model Answer: Arthur's Magazine — it began in 1844, while First for Women was started in 1989.
📚 Ground Truth: Arthur's Magazine
------------------------------


  4%|▍         | 2/50 [00:08<03:13,  4.04s/it]


[第 2 题]
🔍 Question: The Oberoi family is part of a hotel company that has a head office in what city?
🤖 Model Answer: Delhi
📚 Ground Truth: Delhi
------------------------------


...


[第 48 题]
🔍 Question: Don Barry Mason was the founder of the Psychedelic Shamanistic Institute (PSI), which other member that's Welsh, that died on 10 April 2016?
🤖 Model Answer: Howard Marks
📚 Ground Truth: Dennis Howard Marks
------------------------------


 98%|█████████▊| 49/50 [04:52<00:04,  4.27s/it]


[第 49 题]
🔍 Question: What male actor starred in The Messenger?
🤖 Model Answer: Robert Sheehan.
📚 Ground Truth: Robert Sheehan
------------------------------


100%|██████████| 50/50 [04:55<00:00,  5.92s/it]


[第 50 题]
🔍 Question: Are Gin and tonic and Paloma both cocktails based on tequila?
🤖 Model Answer: No. The Paloma is tequila-based, while the Gin and Tonic is made with gin (not tequila).
📚 Ground Truth: no
------------------------------

✅ 评估数据准备完毕，eval_dataset 已生成。


In [ ]:
# ==========================================
# 第六步：修复评估流程 (Split Roles)
# ==========================================

from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
    answer_correctness,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# -----------------------------------------------------------
# 关键修复：定义两个不同的 LLM
# 1. 你的 RAG 系统依然使用 GPT-5 mini (刚才已经跑完了，结果存在 eval_dataset 里了)
# 2. 但我们需要一个支持 temperature=0 的“传统模型”来做裁判
# -----------------------------------------------------------

# 定义专门用于评估的“裁判模型” (使用 gpt-4o 或 gpt-3.5-turbo)
# 它们支持 temperature=0，不会报错
judge_llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 包装裁判模型
ragas_judge_llm = LangchainLLMWrapper(judge_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# 定义指标列表
metrics = [
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
    answer_correctness,
]

print("正在使用 GPT-4o 作为裁判进行评估 (这不会影响 GPT-5 mini 生成的答案)...")


# 运行评估
results = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=ragas_judge_llm,      # <--- 这里用裁判模型
    embeddings=ragas_embeddings
)

# 展示结果
print("\n========== RAGAS 评估结果 ==========")
print(results)

# 转换为 Pandas DataFrame
df_results = results.to_pandas()
# 直接显示所有行，不再用 head() 截断
df_results

正在使用 GPT-4o 作为裁判进行评估 (这不会影响 GPT-5 mini 生成的答案)...


/tmp/ipython-input-4088919701.py:6: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipython-input-4088919701.py:6: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
/tmp/ipython-input-4088919701.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipython-input-4088919701.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1

Evaluating:   0%|          | 0/250 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[31]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-IsBFieVFCuY8zx7uwf6w9Q7h on tokens per min (TPM): Limit 30000, Used 29581, Requested 1806. Please try again in 2.774s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
ERROR:ragas.executor:Exception raised in Job[24]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-IsBFieVFCuY8zx7uwf6w9Q7h on tokens per min (TPM): Limit 30000, Used 29872, Requested 1350. Please try again in 2.444s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
ERROR:ragas.executor:Exception raised in Job[26]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[32]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[3

KeyboardInterrupt: 

In [ ]:
csv_filename = "ragas_evaluation_results_Raw_RAG.csv"
df_results.to_csv(csv_filename, index=False, encoding='utf-8-sig')

## Agentic RAG


## Version 2 Agent, critic, director, information extrator

In [ ]:
import json
from typing import List, Dict

!pip install -q sentence-transformers

# --- 1. 模型与核心组件 (改为 langchain_core) ---
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage # 替换了 langchain.schema

# --- 2. 社区组件 (保持现状，看起来是对的) ---
# 注意：cross_encoders 属于社区包，这个路径是对的
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# ================= 1. 初始化模型 =================

# --- LLM 设置 (LLM 1, 2, 3 都复用这个，也可以实例化多个不同能力的模型) ---
# 建议使用 GPT-4o 或 GPT-4-turbo 以保证 Router 和 Critic 的逻辑能力
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

# --- Reranker 设置 (用于从 HNSW 的结果中筛选 Top 20) ---
# 使用 BAAI 的 Reranker，效果比普通 Embedding 相似度更准
print("正在加载 Reranker 模型 (BAAI/bge-reranker-base)...")
reranker = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

# ================= 2. 定义 Prompt 模板 =================

# [LLM 1] Router & Query Rewriter
# 职责：判断信息是否足够，如果不够，生成新的搜索词
ROUTER_PROMPT = """
You are an expert multi-hop reasoning agent. Your task is to answer the user's question based on the provided context.
Current Context:
{context_summary}

User Question: {question}

Analyze the current context. Do you have enough information to answer the question concretely?
- If YES: Return a JSON with "action": "answer" and the "final_answer".
- If NO: Identify what information is missing. Return a JSON with "action": "search" and a "new_search_query".
  - The "new_search_query" MUST be a specific, entity-centric query optimized for a Wikipedia search (e.g., "Director of movie Matrix").
  - DO NOT ask complex questions. Break it down.

Format: JSON only.
"""

# [LLM 2] Critic / Filter
# 职责：从 20 个文档中选出真正有用的 Top 5 (Batch Process)
CRITIC_PROMPT = """
You are a strict data filter. I will provide a list of retrieved documents.
Your goal is to select the top {k} documents that are MOST relevant and helpful to answer the query: "{query}".

Documents:
{documents_text}

Criteria:
1. Must contain specific entities or facts related to the query.
2. Filter out documents that just share keywords but have the wrong logic (Distractors).

Output format: Return ONLY a JSON list of the INDICES of the selected documents. Example: [0, 4, 12]
"""

# [LLM 3] Information Extractor (Not Summary)
# 职责：当文档过多时，抽取实体和关系，压缩上下文
EXTRACTOR_PROMPT = """
The context has become too long. Your task is to perform "Information Extraction".
Read the following text and extract ALL key entities, dates, relationships, and facts relevant to the User's original goal.

Guidelines:
1. DO NOT summarize into a story. List facts.
2. KEEP specific names, dates, and numbers exactly as they appear.
3. Discard fluff and general descriptions.

Text to process:
{long_context}
"""

# ================= 3. 定义 Agent 类 =================

class AgenticRagSystem:
    def __init__(self, vectorstore, llm, reranker):
        self.vectorstore = vectorstore
        self.llm = llm
        self.reranker = reranker
        self.context_memory = [] # 存储通过筛选的文档内容
        self.max_iterations = 9  # 防止死循环

    def format_docs(self, docs):
        return "\n\n".join([f"[Doc {i}] {d.page_content}" for i, d in enumerate(docs)])

    def retrieve_and_rerank(self, query, top_k_initial=50, top_k_final=20):
        print(f"   🔍 [Retrieve] HNSW Searching for: '{query}'...")
        # 1. HNSW 粗排 (Recall)
        initial_docs = self.vectorstore.similarity_search(query, k=top_k_initial)

        # 2. Cross-Encoder 重排序 (Precision)
        # Reranker 需要 (Query, Doc) 对
        pairs = [[query, d.page_content] for d in initial_docs]
        scores = self.reranker.score(pairs)

        # 根据分数排序并取 Top 20
        sorted_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        top_docs = [initial_docs[i] for i in sorted_indices[:top_k_final]]

        print(f"   📉 [Rerank] Filtered {top_k_initial} -> {top_k_final} docs.")
        return top_docs

    def llm_critic_filter(self, query, docs, select_k=5):
        print(f"   🧠 [LLM 2 Critic] Batch processing {len(docs)} docs...")
        docs_text = self.format_docs(docs)

        msg = ChatPromptTemplate.from_template(CRITIC_PROMPT).format_messages(
            k=select_k, query=query, documents_text=docs_text
        )

        response = self.llm.invoke(msg).content
        try:
            # 尝试解析 JSON 索引列表
            import re
            # 简单的正则提取 list 部分，防止 LLM 说废话
            match = re.search(r'\[.*\]', response, re.DOTALL)
            if match:
                indices = json.loads(match.group())
                selected_docs = [docs[i] for i in indices if i < len(docs)]
                print(f"   ✅ [LLM 2 Critic] Selected {len(selected_docs)} high-quality docs.")
                return selected_docs
        except:
            print("   ⚠️ [Critic Error] Failed to parse JSON. Fallback to Top 3.")
            return docs[:3]
        return docs[:select_k] # Fallback

    def llm_compressor(self):
        print("   🗜️ [LLM 3 Compressor] Context > 15 docs. Extracting Information...")
        full_text = "\n".join(self.context_memory)

        msg = ChatPromptTemplate.from_template(EXTRACTOR_PROMPT).format_messages(
            long_context=full_text
        )

        extracted_facts = self.llm.invoke(msg).content
        # 压缩后替换原 Memory
        self.context_memory = [extracted_facts]
        print("   ✅ Context compressed into key facts.")

    def run(self, user_question):
        current_context = ""
        iteration = 0

        print(f"🚀 Start Agentic RAG for: {user_question}")

        while iteration < self.max_iterations:
            print(f"\n--- Iteration {iteration + 1} ---")

            # --- Step 1: LLM 1 Router ---
            # 构造当前已知信息的摘要
            context_str = "\n".join(self.context_memory) if self.context_memory else "No external information yet."

            msg = ChatPromptTemplate.from_template(ROUTER_PROMPT).format_messages(
                context_summary=context_str, question=user_question
            )

            # 强制 JSON 模式 (如果模型支持，否则依赖 Prompt)
            router_response = self.llm.invoke(msg).content

            try:
                # 清洗 markdown 代码块
                router_json = json.loads(router_response.replace("```json", "").replace("```", ""))
            except:
                print("Router JSON parsing failed. Ending.")
                break

            action = router_json.get("action")

            if action == "answer":
                print(f"🎉 [LLM 1] Enough Info! Generating Answer.")
                return router_json.get("final_answer")

            elif action == "search":
                new_query = router_json.get("new_search_query")
                print(f"🤔 [LLM 1] Missing Info. Decided to search: '{new_query}'")

                # --- Step 2: Retrieve & Rerank ---
                # HNSW (50) -> Rerank (20)
                top_20_docs = self.retrieve_and_rerank(new_query, top_k_initial=50, top_k_final=20)

                # --- Step 3: LLM 2 Critic ---
                # LLM Filter (20 -> 5)
                # 注意：这里我们把新 Query 传给 Critic，而不是原 User Question，保证相关性
                final_docs = self.llm_critic_filter(new_query, top_20_docs, select_k=5)

                # Update Memory
                for d in final_docs:
                    self.context_memory.append(d.page_content)

                # --- Step 4: LLM 3 Check Context Size ---
                # 如果累积了太多文档片段（比如 > 15），触发信息抽取
                if len(self.context_memory) > 15:
                    self.llm_compressor()

            iteration += 1

        return "Max iterations reached. Could not find complete answer."

# ================= 4. 运行测试 =================

# 实例化系统
# 注意：这里的 vectorstore 来自你上一段代码构建的 Chroma 实例
agent = AgenticRagSystem(vectorstore, llm, reranker)

# 测试一个多跳问题
# 示例：HotpotQA 经典问题
# 假设问题是： "Which magazine was founded first, Arthur's Magazine or First for Women?"
# 这需要先搜 Arthur's Magazine 的时间，再搜 First for Women 的时间，最后比较。

# 我们可以直接用数据集里的问题来测
test_idx = 4
test_q = dataset[test_idx]['question']


final_answer = agent.run(test_q)

print(f"\n================ FINAL ANSWER ================")
print(final_answer)

正在加载 Reranker 模型 (BAAI/bge-reranker-base)...


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
# ================= 5. 运行评测循环 (Evaluation Loop) =================

# 准备测试数据
EVAL_SIZE = 50
test_dataset = dataset.select(range(EVAL_SIZE))

ragas_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

print(f"\n🚀 开始批量评测 (共 {EVAL_SIZE} 条)...")

for i, row in enumerate(tqdm(test_dataset)):
    question = row['question']
    ground_truth = row['answer']

    # [关键步骤] 1. 清空 Agent 的记忆，防止上一题的文档干扰这一题
    agent.context_memory = []

    # [关键步骤] 2. 运行 Agent
    try:
        model_answer = agent.run(question)
    except Exception as e:
        model_answer = f"Error: {str(e)}"

    # [关键步骤] 3. 获取 Agent 最终使用的上下文 (用于 RAGAS 评分)
    final_contexts = agent.context_memory.copy()

    # --- 实时日志打印 (已修改：每一题都打印，包含 Gold Answer) ---
    print(f"\n[Case {i+1}]")
    print(f"🔍 Question:  {question}")
    print(f"🤖 Agent Ans: {model_answer}")
    print(f"🎯 Gold Ans:  {ground_truth}")   # 新增这一行
    print(f"📄 Docs Used: {len(final_contexts)}")
    print("-" * 50)

    # --- 收集数据 ---
    ragas_data["question"].append(question)
    ragas_data["answer"].append(model_answer)
    ragas_data["contexts"].append(final_contexts)
    ragas_data["ground_truth"].append(ground_truth)

print("\n✅ 所有推理完成。")

# ================= 6. 生成 RAGAS 数据集 =================
eval_dataset_agent = Dataset.from_dict(ragas_data)
print(f"Eval Dataset 生成完毕，包含 {len(eval_dataset)} 条记录。")

In [ ]:
from datasets import Dataset
eval_dataset_agent = Dataset.from_dict(ragas_data)
print(f"Eval Dataset 生成完毕，包含 {len(eval_dataset_agent)} 条记录。")

In [ ]:
# ==========================================
# 第六步：修复评估流程 (Split Roles)
# ==========================================

from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# -----------------------------------------------------------
# 关键修复：定义两个不同的 LLM
# 1. 你的 RAG 系统依然使用 GPT-5 mini (刚才已经跑完了，结果存在 eval_dataset 里了)
# 2. 但我们需要一个支持 temperature=0 的“传统模型”来做裁判
# -----------------------------------------------------------

# 定义专门用于评估的“裁判模型” (使用 gpt-4o 或 gpt-3.5-turbo)
# 它们支持 temperature=0，不会报错
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 包装裁判模型
ragas_judge_llm = LangchainLLMWrapper(judge_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# 定义指标列表
metrics = [
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
]

print("正在使用 GPT-4o 作为裁判进行评估 (这不会影响 GPT-5 mini 生成的答案)...")


# 运行评估
results = evaluate(
    dataset=eval_dataset_agent,
    metrics=metrics,
    llm=ragas_judge_llm,      # <--- 这里用裁判模型
    embeddings=ragas_embeddings
)

# 展示结果
print("\n========== RAGAS 评估结果 ==========")
print(results)

# 转换为 Pandas DataFrame
df_results = results.to_pandas()
# 直接显示所有行，不再用 head() 截断
df_results

## Version 3 , add in-context learning: fail case

In [ ]:
import json
import re
from typing import List, Dict

# !pip install -q sentence-transformers
# 假设环境已安装好

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# ================= 1. 初始化模型 =================

llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

print("正在加载 Reranker 模型 (BAAI/bge-reranker-base)...")
reranker = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

# ================= 2. 定义 Prompt 模板 (已修改支持负反馈) =================

# [LLM 1] Router & Query Rewriter (Updated with Negative Feedback)
ROUTER_PROMPT = """
You are an expert multi-hop reasoning agent. Your task is to answer the user's question based on the provided context.

User Question: {question}

--- Context Status ---
Current Retrieved Knowledge:
{context_summary}

--- Search History & Negative Feedback ---
The following queries have already been tried and FAILED to yield useful results.
DO NOT repeat these exact queries:
{failed_searches}

--- Decision ---
Analyze the current context. Do you have enough information to answer the question concretely?
- If YES: Return a JSON with "action": "answer" and the "final_answer".
- If NO: Identify what information is missing. Return a JSON with "action": "search" and a "new_search_query".
  - **CRITICAL**: Since you are seeing the "Search History", if previous searches failed, you MUST generate a DIFFERENT query strategy (e.g., use synonyms, broader terms, or search for a related entity).
  - The "new_search_query" MUST be a specific, entity-centric query optimized for a Wikipedia search.

Format: JSON only.
"""

# [LLM 2] Critic / Filter (保持不变)
CRITIC_PROMPT = """
You are a strict data filter. I will provide a list of retrieved documents.
Your goal is to select the top {k} documents that are MOST relevant and helpful to answer the query: "{query}".

Documents:
{documents_text}

Criteria:
1. Must contain specific entities or facts related to the query.
2. Filter out documents that just share keywords but have the wrong logic (Distractors).

Output format: Return ONLY a JSON list of the INDICES of the selected documents. Example: [0, 4, 12]
"""

# [LLM 3] Information Extractor (保持不变)
EXTRACTOR_PROMPT = """
The context has become too long. Your task is to perform "Information Extraction".
Read the following text and extract ALL key entities, dates, relationships, and facts relevant to the User's original goal.

Guidelines:
1. DO NOT summarize into a story. List facts.
2. KEEP specific names, dates, and numbers exactly as they appear.
3. Discard fluff and general descriptions.

Text to process:
{long_context}
"""

# ================= 3. 定义 Agent 类 (添加负反馈逻辑) =================

class AgenticRagSystem_3:
    def __init__(self, vectorstore, llm, reranker):
        self.vectorstore = vectorstore
        self.llm = llm
        self.reranker = reranker
        self.context_memory = []
        self.max_iterations = 9
        # 新增：虽然 run 里面会重置，但在 init 定义是个好习惯
        self.failed_searches = []

    def format_docs(self, docs):
        return "\n\n".join([f"[Doc {i}] {d.page_content}" for i, d in enumerate(docs)])

    def retrieve_and_rerank(self, query, top_k_initial=50, top_k_final=20):
        print(f"   🔍 [Retrieve] HNSW Searching for: '{query}'...")
        initial_docs = self.vectorstore.similarity_search(query, k=top_k_initial)

        if not initial_docs:
            return []

        pairs = [[query, d.page_content] for d in initial_docs]
        scores = self.reranker.score(pairs)

        sorted_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        top_docs = [initial_docs[i] for i in sorted_indices[:top_k_final]]

        print(f"   📉 [Rerank] Filtered {top_k_initial} -> {len(top_docs)} docs.")
        return top_docs

    def llm_critic_filter(self, query, docs, select_k=5):
        if not docs:
            return []

        print(f"   🧠 [LLM 2 Critic] Batch processing {len(docs)} docs...")
        docs_text = self.format_docs(docs)

        msg = ChatPromptTemplate.from_template(CRITIC_PROMPT).format_messages(
            k=select_k, query=query, documents_text=docs_text
        )

        response = self.llm.invoke(msg).content
        try:
            match = re.search(r'\[.*\]', response, re.DOTALL)
            if match:
                indices = json.loads(match.group())
                selected_docs = [docs[i] for i in indices if i < len(docs)]
                print(f"   ✅ [LLM 2 Critic] Selected {len(selected_docs)} high-quality docs.")
                return selected_docs
        except:
            print("   ⚠️ [Critic Error] Failed to parse JSON. Fallback to Top 3.")
            return docs[:3]
        return docs[:select_k]

    def llm_compressor(self):
        print("   🗜️ [LLM 3 Compressor] Context > 15 docs. Extracting Information...")
        full_text = "\n".join(self.context_memory)
        msg = ChatPromptTemplate.from_template(EXTRACTOR_PROMPT).format_messages(
            long_context=full_text
        )
        extracted_facts = self.llm.invoke(msg).content
        self.context_memory = [extracted_facts]
        print("   ✅ Context compressed into key facts.")

    def run(self, user_question):
        # 每次运行前重置状态
        self.context_memory = []
        self.failed_searches = [] # 重置失败记录

        print(f"🚀 Start Agentic RAG for: {user_question}")

        iteration = 0
        while iteration < self.max_iterations:
            print(f"\n--- Iteration {iteration + 1} ---")

            # 1. 准备 Context 和 失败历史
            context_str = "\n".join(self.context_memory) if self.context_memory else "No external information yet."

            if self.failed_searches:
                failed_str = "\n".join([f"- {item}" for item in self.failed_searches])
            else:
                failed_str = "None."

            # 2. Router 决策
            # 注意：这里把 failed_searches 传给 Prompt
            msg = ChatPromptTemplate.from_template(ROUTER_PROMPT).format_messages(
                context_summary=context_str,
                failed_searches=failed_str,
                question=user_question
            )

            router_response = self.llm.invoke(msg).content

            try:
                router_json = json.loads(router_response.replace("```json", "").replace("```", ""))
            except:
                print("Router JSON parsing failed. Ending.")
                break

            action = router_json.get("action")

            if action == "answer":
                print(f"🎉 [LLM 1] Enough Info! Generating Answer.")
                return router_json.get("final_answer")

            elif action == "search":
                new_query = router_json.get("new_search_query")

                # [Negative Feedback Check 1] 简单的本地去重保护
                # 如果 LLM 忽略了 Prompt 里的警告，再次生成了完全一样的失败 Query
                is_duplicate = any(new_query in s for s in self.failed_searches)
                if is_duplicate:
                    print(f"⚠️ [Loop Guard] LLM repeated a failed query: '{new_query}'. Modifying it.")
                    new_query = new_query + " details wikipedia" # 强制加点噪音试图改变结果

                print(f"🤔 [LLM 1] Missing Info. Decided to search: '{new_query}'")

                # 3. 检索
                top_20_docs = self.retrieve_and_rerank(new_query, top_k_initial=50, top_k_final=20)

                # 4. Critic 过滤
                final_docs = self.llm_critic_filter(new_query, top_20_docs, select_k=5)

                # [Negative Feedback Loop Implementation]
                # 核心逻辑：如果过滤后文档数为0，或者检索本身就是空的 -> 视为失败
                if len(final_docs) == 0:
                    print(f"❌ [Failure Feedback] Query '{new_query}' returned 0 useful docs.")
                    # 记录失败原因，供下一轮 Router 参考
                    self.failed_searches.append(f"Query '{new_query}' -> Result: No relevant documents found.")

                    # 关键：不要把空内容加到 memory，也不要增加 iteration 计数太多
                    # 直接 continue，跳回 Router，让它看到这个失败记录并重写 Query
                    iteration += 1
                    continue
                else:
                    # 成功：更新 Context
                    print(f"✅ [Success] Added {len(final_docs)} docs to context.")
                    for d in final_docs:
                        self.context_memory.append(d.page_content)

                # Context 压缩检查
                if len(self.context_memory) > 15:
                    self.llm_compressor()

            iteration += 1

        return "Max iterations reached. Could not find complete answer."

In [ ]:
# ================= 5. 运行评测循环 (Evaluation Loop) =================

agent = AgenticRagSystem_3(vectorstore, llm, reranker)

# 准备测试数据
EVAL_SIZE = 50
test_dataset = dataset.select(range(EVAL_SIZE))

ragas_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

print(f"\n🚀 开始批量评测 (共 {EVAL_SIZE} 条)...")

for i, row in enumerate(tqdm(test_dataset)):
    question = row['question']
    ground_truth = row['answer']

    # [关键步骤] 1. 清空 Agent 的记忆，防止上一题的文档干扰这一题
    agent.context_memory = []

    # [关键步骤] 2. 运行 Agent
    try:
        model_answer = agent.run(question)
    except Exception as e:
        model_answer = f"Error: {str(e)}"

    # [关键步骤] 3. 获取 Agent 最终使用的上下文 (用于 RAGAS 评分)
    final_contexts = agent.context_memory.copy()

    # --- 实时日志打印 (已修改：每一题都打印，包含 Gold Answer) ---
    print(f"\n[Case {i+1}]")
    print(f"🔍 Question:  {question}")
    print(f"🤖 Agent Ans: {model_answer}")
    print(f"🎯 Gold Ans:  {ground_truth}")   # 新增这一行
    print(f"📄 Docs Used: {len(final_contexts)}")
    print("-" * 50)

    # --- 收集数据 ---
    ragas_data["question"].append(question)
    ragas_data["answer"].append(model_answer)
    ragas_data["contexts"].append(final_contexts)
    ragas_data["ground_truth"].append(ground_truth)

print("\n✅ 所有推理完成。")



In [ ]:
# ================= 6. 生成 RAGAS 数据集 =================
eval_dataset_agent = Dataset.from_dict(ragas_data)
print(f"Eval Dataset 生成完毕，包含 {len(eval_dataset)} 条记录。")

# Version 4, add planning: for multi-hop Vector search


In [ ]:
import json
import re
from typing import List, Dict

# !pip install -q sentence-transformers
# 假设环境已安装好

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# ================= 1. 初始化模型 =================

llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

print("正在加载 Reranker 模型 (BAAI/bge-reranker-base)...")
reranker = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

# ================= 2. 定义 Prompt 模板 =================

# [LLM 1] Router & Planner (Fix: 引入全量历史 + 原子化约束)
ROUTER_PROMPT = """
You are an expert reasoning agent.

User Question: {question}

--- Context (What we know) ---
{context_summary}

--- Search History (Queries already tried) ---
The following queries have been executed but did NOT yield the final answer.
**CRITICAL: DO NOT REPEAT ANY QUERY IN THIS LIST.**
{search_history}

--- Planning Instructions ---
1. **Analyze:** Is the question asking about a specific attribute (e.g., age, nationality) of an entity you haven't identified yet?
2. **Decompose:** If YES, do NOT search for the final attribute directly. First, search for the **missing entity**.
3. **Refine:** Check the "Search History". If you intended to search "X", but "X" is in the history, you MUST change your keywords (e.g., use "X biography", "X marriage", "X family").

--- Constraints (MUST FOLLOW) ---
- **Atomic Queries Only:** Your search query must focus on ONE unknown fact at a time.
- **No Compound Queries:** Do not search for "Entity + Attribute" (e.g., "wife nationality") unless you already know the Entity's name.

--- Few-Shot Examples ---
User: "What nationality was James Henry Miller's wife?"
Context: Empty.
Output: {{
  "thought_process": "I need to find the name of James Henry Miller's wife first. Searching for 'wife nationality' directly is forbidden. I haven't searched for his name yet.",
  "new_search_query": "James Henry Miller wife name",
  "action": "search"
}}

User: "What nationality was James Henry Miller's wife?"
Context: "James Henry Miller married Patricia Smith."
Output: {{
  "thought_process": "I now know the wife is Patricia Smith. Now I can search for her nationality.",
  "new_search_query": "Patricia Smith nationality",
  "action": "search"
}}

--- Your Turn ---
Format: JSON only. Keys: "thought_process", "action", "new_search_query" (or "final_answer").
"""

# [LLM 2] Critic / Filter (保持不变)
CRITIC_PROMPT = """
You are a strict data filter. I will provide a list of retrieved documents.
Your goal is to select the top {k} documents that are MOST relevant and helpful to answer the query: "{query}".

Documents:
{documents_text}

Criteria:
1. Must contain specific entities or facts related to the query.
2. Filter out documents that just share keywords but have the wrong logic (Distractors).

Output format: Return ONLY a JSON list of the INDICES of the selected documents. Example: [0, 4, 12]
"""

# [LLM 3] Information Extractor (保持不变)
EXTRACTOR_PROMPT = """
The context has become too long. Your task is to perform "Information Extraction".
Read the following text and extract ALL key entities, dates, relationships, and facts relevant to the User's original goal.

Guidelines:
1. DO NOT summarize into a story. List facts.
2. KEEP specific names, dates, and numbers exactly as they appear.

Text to process:
{long_context}
"""

# ================= 3. 定义 Agent 类 (Fix: 强制去重逻辑) =================

class AgenticRagSystem_4:
    def __init__(self, vectorstore, llm, reranker):
        self.vectorstore = vectorstore
        self.llm = llm
        self.reranker = reranker
        self.context_memory = []
        self.max_iterations = 9
        # 全量搜索历史，用于去重
        self.search_history = set()

    def format_docs(self, docs):
        return "\n\n".join([f"[Doc {i}] {d.page_content}" for i, d in enumerate(docs)])

    def retrieve_and_rerank(self, query, top_k_initial=50, top_k_final=20):
        print(f"   🔍 [Retrieve] HNSW Searching for: '{query}'...")
        initial_docs = self.vectorstore.similarity_search(query, k=top_k_initial)
        if not initial_docs: return []

        pairs = [[query, d.page_content] for d in initial_docs]
        scores = self.reranker.score(pairs)
        sorted_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        top_docs = [initial_docs[i] for i in sorted_indices[:top_k_final]]
        print(f"   📉 [Rerank] Filtered {top_k_initial} -> {len(top_docs)} docs.")
        return top_docs

    def llm_critic_filter(self, query, docs, select_k=5):
        if not docs: return []
        print(f"   🧠 [LLM 2 Critic] Batch processing {len(docs)} docs...")
        docs_text = self.format_docs(docs)

        msg = ChatPromptTemplate.from_template(CRITIC_PROMPT).format_messages(
            k=select_k, query=query, documents_text=docs_text
        )
        response = self.llm.invoke(msg).content
        try:
            match = re.search(r'\[.*\]', response, re.DOTALL)
            if match:
                indices = json.loads(match.group())
                selected = [docs[i] for i in indices if i < len(docs)]
                print(f"   ✅ [LLM 2 Critic] Selected {len(selected)} high-quality docs.")
                return selected
        except:
            return docs[:3]
        return docs[:select_k]

    def llm_compressor(self):
        print("   🗜️ [LLM 3 Compressor] Context > 15 docs. Extracting Information...")
        full_text = "\n".join(self.context_memory)
        msg = ChatPromptTemplate.from_template(EXTRACTOR_PROMPT).format_messages(long_context=full_text)
        extracted_facts = self.llm.invoke(msg).content
        self.context_memory = [extracted_facts]
        print("   ✅ Context compressed.")

    def run(self, user_question):
        # 重置状态
        self.context_memory = []
        self.search_history = set()

        print(f"🚀 Start Agentic RAG for: {user_question}")

        iteration = 0
        while iteration < self.max_iterations:
            print(f"\n--- Iteration {iteration + 1} ---")

            context_str = "\n".join(self.context_memory) if self.context_memory else "No external information yet."

            # 1. 构造 Search History 字符串
            if self.search_history:
                history_str = "\n".join([f"- {q}" for q in self.search_history])
            else:
                history_str = "None."

            # 2. Router 决策
            msg = ChatPromptTemplate.from_template(ROUTER_PROMPT).format_messages(
                context_summary=context_str,
                search_history=history_str, # 传入全量历史
                question=user_question
            )

            router_response = self.llm.invoke(msg).content

            try:
                router_json = json.loads(router_response.replace("```json", "").replace("```", ""))
            except:
                print("Router JSON parsing failed. Ending.")
                break

            thought = router_json.get("thought_process", "No thought provided.")
            print(f"   💭 [Thought]: {thought}")

            action = router_json.get("action")

            if action == "answer":
                print(f"🎉 [LLM 1] Enough Info! Generating Answer.")
                return router_json.get("final_answer")

            elif action == "search":
                new_query = router_json.get("new_search_query")

                # ============================================================
                # 🔥 关键修复：全量历史去重与强制改写 🔥
                # ============================================================
                if new_query in self.search_history:
                    print(f"⚠️ [Loop Detected] LLM repeated query '{new_query}' despite instructions.")

                    # 强制变策列表 (Strategy Shift)
                    strategies = [" marriage", " spouse", " family", " biography", " personal life", " children"]
                    # 简单的轮询机制，根据迭代次数选择不同的后缀
                    suffix = strategies[iteration % len(strategies)]

                    original_query = new_query
                    new_query = f"{original_query} {suffix}"
                    print(f"🔀 [Force Modify] Changed query to: '{new_query}'")

                # 无论是否修改，都将新 Query 加入历史，防止下一轮又生成这个
                self.search_history.add(new_query)

                print(f"🤔 [LLM 1] Decided to search: '{new_query}'")

                # 3. 执行检索
                top_20_docs = self.retrieve_and_rerank(new_query, top_k_initial=50, top_k_final=20)

                # 4. Critic 过滤
                final_docs = self.llm_critic_filter(new_query, top_20_docs, select_k=5)

                if len(final_docs) == 0:
                    print(f"❌ [Failure] Query returned 0 useful docs.")
                    # 此时不 add 到 context，直接进入下一轮 Router
                    # Router 会在 search_history 里看到这个失败的 query，尝试别的路径
                    iteration += 1
                    continue
                else:
                    print(f"✅ [Success] Added {len(final_docs)} docs to context.")
                    for d in final_docs:
                        self.context_memory.append(d.page_content)

                if len(self.context_memory) > 15:
                    self.llm_compressor()

            iteration += 1

        return "Max iterations reached."

In [ ]:
agent_4 = AgenticRagSystem_4(vectorstore, llm, reranker)

# 测试一个多跳问题
# 示例：HotpotQA 经典问题
# 假设问题是： "Which magazine was founded first, Arthur's Magazine or First for Women?"
# 这需要先搜 Arthur's Magazine 的时间，再搜 First for Women 的时间，最后比较。

# 我们可以直接用数据集里的问题来测
test_idx = 3
test_q = dataset[test_idx]['question']


final_answer = agent_4.run(test_q)

In [ ]:
# ================= 5. 运行评测循环 (Evaluation Loop) =================

# 准备测试数据
EVAL_SIZE = 50
test_dataset = dataset.select(range(EVAL_SIZE))

ragas_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

print(f"\n🚀 开始批量评测 (共 {EVAL_SIZE} 条)...")

for i, row in enumerate(tqdm(test_dataset)):
    question = row['question']
    ground_truth = row['answer']

    # [关键步骤] 1. 清空 Agent 的记忆，防止上一题的文档干扰这一题
    agent.context_memory = []

    # [关键步骤] 2. 运行 Agent
    try:
        model_answer = agent_4.run(question)
    except Exception as e:
        model_answer = f"Error: {str(e)}"

    # [关键步骤] 3. 获取 Agent 最终使用的上下文 (用于 RAGAS 评分)
    final_contexts = agent.context_memory.copy()

    # --- 实时日志打印 (已修改：每一题都打印，包含 Gold Answer) ---
    print(f"\n[Case {i+1}]")
    print(f"🔍 Question:  {question}")
    print(f"🤖 Agent Ans: {model_answer}")
    print(f"🎯 Gold Ans:  {ground_truth}")   # 新增这一行
    print(f"📄 Docs Used: {len(final_contexts)}")
    print("-" * 50)

    # --- 收集数据 ---
    ragas_data["question"].append(question)
    ragas_data["answer"].append(model_answer)
    ragas_data["contexts"].append(final_contexts)
    ragas_data["ground_truth"].append(ground_truth)

print("\n✅ 所有推理完成。")

# ================= 6. 生成 RAGAS 数据集 =================
eval_dataset_agent = Dataset.from_dict(ragas_data)
print(f"Eval Dataset 生成完毕，包含 {len(eval_dataset)} 条记录。")

In [ ]:
# ==========================================
# 第六步：修复评估流程 (Split Roles)
# ==========================================

from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
    answer_correctness,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# -----------------------------------------------------------
# 关键修复：定义两个不同的 LLM
# 1. 你的 RAG 系统依然使用 GPT-5 mini (刚才已经跑完了，结果存在 eval_dataset 里了)
# 2. 但我们需要一个支持 temperature=0 的“传统模型”来做裁判
# -----------------------------------------------------------

# 定义专门用于评估的“裁判模型” (使用 gpt-4o 或 gpt-3.5-turbo)
# 它们支持 temperature=0，不会报错
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 包装裁判模型
ragas_judge_llm = LangchainLLMWrapper(judge_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# 定义指标列表
metrics = [
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
    answer_correctness,
]

print("正在使用 GPT-4o 作为裁判进行评估 (这不会影响 GPT-5 mini 生成的答案)...")


# 运行评估
results = evaluate(
    dataset=eval_dataset_agent,
    metrics=metrics,
    llm=ragas_judge_llm,      # <--- 这里用裁判模型
    embeddings=ragas_embeddings
)

# 展示结果
print("\n========== RAGAS 评估结果 ==========")
print(results)

# 转换为 Pandas DataFrame
df_results = results.to_pandas()
# 直接显示所有行，不再用 head() 截断
df_results

# Version 5: Reslove the deadlock

In [ ]:
import json
import re
from typing import List, Dict

# 假设环境已安装好所有依赖
# !pip install -q sentence-transformers langchain_openai langchain_community

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# ================= 1. 初始化模型 =================

# 建议：如果显存允许，Router 用 GPT-4o 或 GPT-5-mini 保证逻辑，Critic 可以用小一点的模型
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

print("正在加载 Reranker 模型 (BAAI/bge-reranker-base)...")
reranker = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

# ================= 2. 定义 Prompt 模板 =================

# [LLM 1] Router & Planner (终极修复版)
# 1. 使用 {{ }} 转义 JSON 示例，防止 KeyError
# 2. 加入 Decision Protocol 解决"只想不说"的问题
# 3. 加入 Constraints 解决"复合查询"的问题
ROUTER_PROMPT = """
You are an expert reasoning agent.

User Question: {question}

--- Context (What we know) ---
{context_summary}

--- Search History (Queries already tried) ---
The following queries have been executed. **CRITICAL: DO NOT REPEAT ANY QUERY IN THIS LIST.**
{search_history}

--- Decision Protocol ---
1. **CHECK ANSWER FIRST:** Look at the "Context". Does it contain ALL the information needed to answer the User Question?
   - **CRITICAL:** If you have the answer (e.g., dates for both items, or the specific name), you MUST return "action": "answer" IMMEDIATELY.
   - DO NOT THINK AGAIN. DO NOT SEARCH AGAIN.
   - If comparing X and Y, and you have data for BOTH, output the answer.

2. **Planning (Only if info is missing):** - Identify missing entities.
   - Formulate atomic queries.

--- Constraints (MUST FOLLOW) ---
- **Atomic Queries Only:** Your search query must focus on ONE unknown fact at a time.
- **No Compound Queries:** Do not search for "Entity + Attribute" (e.g., "wife nationality") unless you already know the Entity's name.

--- Few-Shot Examples ---
User: "Which magazine was founded first, A or B?"
Context: "A was founded in 1844. B was founded in 1989."
Output: {{
  "thought_process": "I have the founding years for both A (1844) and B (1989). I can compare them directly. No further search is needed.",
  "action": "answer",
  "final_answer": "A was started first (1844), while B began in 1989."
}}

User: "What nationality was James Henry Miller's wife?"
Context: Empty.
Output: {{
  "thought_process": "I need to find the name of James Henry Miller's wife first. Searching for 'wife nationality' directly is forbidden. I haven't searched for his name yet.",
  "new_search_query": "James Henry Miller wife name",
  "action": "search"
}}

--- Your Turn ---
Format: JSON only. Keys: "thought_process", "action", "new_search_query" OR "final_answer".
"""

# [LLM 2] Critic / Filter (保持不变)
CRITIC_PROMPT = """
You are a strict data filter. I will provide a list of retrieved documents.
Your goal is to select the top {k} documents that are MOST relevant and helpful to answer the query: "{query}".

Documents:
{documents_text}

Criteria:
1. Must contain specific entities or facts related to the query.
2. Filter out documents that just share keywords but have the wrong logic (Distractors).

Output format: Return ONLY a JSON list of the INDICES of the selected documents. Example: [0, 4, 12]
"""

# [LLM 3] Information Extractor (保持不变)
EXTRACTOR_PROMPT = """
The context has become too long. Your task is to perform "Information Extraction".
Read the following text and extract ALL key entities, dates, relationships, and facts relevant to the User's original goal.

Guidelines:
1. DO NOT summarize into a story. List facts.
2. KEEP specific names, dates, and numbers exactly as they appear.

Text to process:
{long_context}
"""

# ================= 3. 定义 Agent 类 =================

class AgenticRagSystem_5:
    def __init__(self, vectorstore, llm, reranker):
        self.vectorstore = vectorstore
        self.llm = llm
        self.reranker = reranker
        self.context_memory = []
        self.max_iterations = 9
        # 全量搜索历史，用于去重
        self.search_history = set()

    def format_docs(self, docs):
        return "\n\n".join([f"[Doc {i}] {d.page_content}" for i, d in enumerate(docs)])

    def retrieve_and_rerank(self, query, top_k_initial=50, top_k_final=20):
        print(f"   🔍 [Retrieve] HNSW Searching for: '{query}'...")
        # 这里的 fetch_k 可以稍微大一点，保证召回率
        initial_docs = self.vectorstore.similarity_search(query, k=top_k_initial)
        if not initial_docs: return []

        pairs = [[query, d.page_content] for d in initial_docs]
        scores = self.reranker.score(pairs)
        sorted_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        top_docs = [initial_docs[i] for i in sorted_indices[:top_k_final]]
        print(f"   📉 [Rerank] Filtered {top_k_initial} -> {len(top_docs)} docs.")
        return top_docs

    def llm_critic_filter(self, query, docs, select_k=5):
        if not docs: return []
        print(f"   🧠 [LLM 2 Critic] Batch processing {len(docs)} docs...")
        docs_text = self.format_docs(docs)

        msg = ChatPromptTemplate.from_template(CRITIC_PROMPT).format_messages(
            k=select_k, query=query, documents_text=docs_text
        )
        response = self.llm.invoke(msg).content
        try:
            match = re.search(r'\[.*\]', response, re.DOTALL)
            if match:
                indices = json.loads(match.group())
                selected = [docs[i] for i in indices if i < len(docs)]
                print(f"   ✅ [LLM 2 Critic] Selected {len(selected)} high-quality docs.")
                return selected
        except:
            return docs[:3]
        return docs[:select_k]

    def llm_compressor(self):
        print("   🗜️ [LLM 3 Compressor] Context > 15 docs. Extracting Information...")
        full_text = "\n".join(self.context_memory)
        msg = ChatPromptTemplate.from_template(EXTRACTOR_PROMPT).format_messages(long_context=full_text)
        extracted_facts = self.llm.invoke(msg).content
        self.context_memory = [extracted_facts]
        print("   ✅ Context compressed.")

    def run(self, user_question):
        # 1. 重置状态
        self.context_memory = []
        self.search_history = set()

        print(f"🚀 Start Agentic RAG for: {user_question}")

        iteration = 0
        while iteration < self.max_iterations:
            print(f"\n--- Iteration {iteration + 1} ---")

            context_str = "\n".join(self.context_memory) if self.context_memory else "No external information yet."

            # 2. 构造 Search History 字符串
            if self.search_history:
                history_str = "\n".join([f"- {q}" for q in self.search_history])
            else:
                history_str = "None."

            # 3. Router 决策
            msg = ChatPromptTemplate.from_template(ROUTER_PROMPT).format_messages(
                context_summary=context_str,
                search_history=history_str,
                question=user_question
            )

            router_response = self.llm.invoke(msg).content

            try:
                router_json = json.loads(router_response.replace("```json", "").replace("```", ""))
            except:
                print("Router JSON parsing failed. Ending.")
                break

            thought = router_json.get("thought_process", "No thought provided.")
            action = router_json.get("action")
            print(f"   💭 [Thought]: {thought}")



            if action == "answer":
                print(f"🎉 [LLM 1] Enough Info! Generating Answer.")
                return router_json.get("final_answer")

            elif action == "search":
                new_query = router_json.get("new_search_query")

                # ---------------------------------------------------------
                # 🔥 [De-Dup] 全量历史去重与强制改写
                # ---------------------------------------------------------
                if new_query in self.search_history:
                    print(f"⚠️ [Loop Detected] LLM repeated query '{new_query}'.")

                    # 强制变策列表
                    strategies = [" marriage", " spouse", " family", " biography", " personal life", " children"]
                    suffix = strategies[iteration % len(strategies)]

                    original_query = new_query
                    new_query = f"{original_query} {suffix}"
                    print(f"🔀 [Force Modify] Changed query to: '{new_query}'")

                # 加入历史
                self.search_history.add(new_query)
                print(f"🤔 [LLM 1] Decided to search: '{new_query}'")

                # 4. 执行检索
                top_20_docs = self.retrieve_and_rerank(new_query, top_k_initial=50, top_k_final=20)

                # 5. Critic 过滤
                final_docs = self.llm_critic_filter(new_query, top_20_docs, select_k=5)

                if len(final_docs) == 0:
                    print(f"❌ [Failure] Query returned 0 useful docs.")
                    # 不更新 Context，直接下一轮，让 Router 看到这个失败记录
                    iteration += 1
                    continue
                else:
                    print(f"✅ [Success] Added {len(final_docs)} docs to context.")
                    for d in final_docs:
                        self.context_memory.append(d.page_content)

                # 6. 压缩上下文
                if len(self.context_memory) > 15:
                    self.llm_compressor()

            iteration += 1

        return "Max iterations reached. Could not find complete answer."

In [ ]:
agent_5 = AgenticRagSystem_5(vectorstore, llm, reranker)

# 测试一个多跳问题
# 示例：HotpotQA 经典问题
# 假设问题是： "Which magazine was founded first, Arthur's Magazine or First for Women?"
# 这需要先搜 Arthur's Magazine 的时间，再搜 First for Women 的时间，最后比较。

# 我们可以直接用数据集里的问题来测
test_idx = 3
test_q = dataset[test_idx]['question']


final_answer = agent_5.run(test_q)

In [ ]:
# ================= 5. 运行评测循环 (Evaluation Loop) =================

# 准备测试数据
EVAL_SIZE = 50
test_dataset = dataset.select(range(EVAL_SIZE))

ragas_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

print(f"\n🚀 开始批量评测 (共 {EVAL_SIZE} 条)...")

for i, row in enumerate(tqdm(test_dataset)):
    question = row['question']
    ground_truth = row['answer']

    # [关键步骤] 1. 清空 Agent 的记忆，防止上一题的文档干扰这一题
    agent.context_memory = []

    # [关键步骤] 2. 运行 Agent
    try:
        model_answer = agent_5.run(question)
    except Exception as e:
        model_answer = f"Error: {str(e)}"

    # [关键步骤] 3. 获取 Agent 最终使用的上下文 (用于 RAGAS 评分)
    final_contexts = agent_5.context_memory.copy()

    # --- 实时日志打印 (已修改：每一题都打印，包含 Gold Answer) ---
    print(f"\n[Case {i+1}]")
    print(f"🔍 Question:  {question}")
    print(f"🤖 Agent Ans: {model_answer}")
    print(f"🎯 Gold Ans:  {ground_truth}")   # 新增这一行
    print(f"📄 Docs Used: {len(final_contexts)}")
    print("-" * 50)

    # --- 收集数据 ---
    ragas_data["question"].append(question)
    ragas_data["answer"].append(model_answer)
    ragas_data["contexts"].append(final_contexts)
    ragas_data["ground_truth"].append(ground_truth)

print("\n✅ 所有推理完成。")



In [ ]:
# ================= 6. 生成 RAGAS 数据集 =================
eval_dataset_agent_5 = Dataset.from_dict(ragas_data)
print(f"Eval Dataset 生成完毕，包含 {len(eval_dataset_agent_5)} 条记录。")

# ==========================================
# 第六步：修复评估流程 (Split Roles)
# ==========================================

from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
    answer_correctness,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# -----------------------------------------------------------
# 关键修复：定义两个不同的 LLM
# 1. 你的 RAG 系统依然使用 GPT-5 mini (刚才已经跑完了，结果存在 eval_dataset 里了)
# 2. 但我们需要一个支持 temperature=0 的“传统模型”来做裁判
# -----------------------------------------------------------

# 定义专门用于评估的“裁判模型” (使用 gpt-4o 或 gpt-3.5-turbo)
# 它们支持 temperature=0，不会报错
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 包装裁判模型
ragas_judge_llm = LangchainLLMWrapper(judge_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# 定义指标列表
metrics = [
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
    answer_correctness,
]

print("正在使用 GPT-4o 作为裁判进行评估 (这不会影响 GPT-5 mini 生成的答案)...")


# 运行评估
results = evaluate(
    dataset=eval_dataset_agent_5,
    metrics=metrics,
    llm=ragas_judge_llm,      # <--- 这里用裁判模型
    embeddings=ragas_embeddings
)

# 展示结果
print("\n========== RAGAS 评估结果 ==========")
print(results)

# 转换为 Pandas DataFrame
df_results = results.to_pandas()
# 直接显示所有行，不再用 head() 截断
df_results

# Version 6 Hybrid Search + HotPot-QA Format alignment

In [ ]:
pip install rank_bm25

In [ ]:
import json
import re
from typing import List, Dict

# 假设环境已安装好所有依赖
# !pip install -q sentence-transformers langchain_openai langchain_community rank_bm25

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
# 【修改点 1】引入 BM25 检索器
from langchain_community.retrievers import BM25Retriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# ================= 1. 初始化模型 =================

llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

print("正在加载 Reranker 模型 (BAAI/bge-reranker-base)...")
reranker = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

# ================= 2. 定义 Prompt 模板 (完全保持不变) =================

ROUTER_PROMPT = """
You are an expert reasoning agent.

User Question: {question}

--- Context (What we know) ---
{context_summary}

--- Search History (Queries already tried) ---
The following queries have been executed. **CRITICAL: DO NOT REPEAT ANY QUERY IN THIS LIST.**
{search_history}

--- Decision Protocol ---
1. **CHECK ANSWER FIRST:** Look at the "Context". Does it contain ALL the information needed to answer the User Question?
   - **CRITICAL:** If you have the answer (e.g., dates for both items, or the specific name), you MUST return "action": "answer" IMMEDIATELY.
   - DO NOT THINK AGAIN. DO NOT SEARCH AGAIN.
   - If comparing X and Y, and you have data for BOTH, output the answer.

   **🔥 ANSWER FORMATTING GUIDELINES (CRITICAL) 🔥**
   - **Direct & Precise:** Stop chatting. Remove all preamble like "The answer is..." or "Based on the context...".
   - **Entity-Centric:** If the question asks "Who", provide the Name. If "When", provide the Date.
   - **Granularity Match:** Mimic the granularity of the Ground Truth.
     - Bad: "The director of the movie Matrix is the Wachowskis." (Too verbose)
     - Good: "The Wachowskis" (Perfect alignment)
   - **Mapping Logic:**
   - **Type: "Which [X] was first...?"** -> Output ONLY the Name of [X]. (e.g., "the Iphone4")
   - **Type: "Who..."** -> Output ONLY the Name. (e.g., "Mike")
   - **Type: "When..."** -> Output ONLY the Date/Year. (e.g., "1989")
   - **Type: "Are/Is/Did..." (Yes/No)** -> Output ONLY "yes" or "no". (Only add a concise correction if absolutely necessary for clarity, e.g., "no").
   - **Type: "What is..."** -> Output ONLY the definition/fact. (e.g., "a tequila-based cocktail")



2. **Planning (Only if info is missing):** - Identify missing entities.
   - Formulate atomic queries.

--- Constraints (MUST FOLLOW) ---
- **Atomic Queries Only:** Your search query must focus on ONE unknown fact at a time.
- **No Compound Queries:** Do not search for "Entity + Attribute" (e.g., "wife nationality") unless you already know the Entity's name.

--- Few-Shot Examples ---
User: "Which phone was founded first, A or B?"
Context: "A was founded in 2001. B was founded in 2007."
Output: {{
  "thought_process": "I have the founding years for both A (1844) and B (1989). I can compare them directly. No further search is needed.",
  "action": "answer",
  "final_answer": "A "
}}

User: "What nationality was James Henry Miller's wife?"
Context: Empty.
Output: {{
  "thought_process": "I need to find the name of James Henry Miller's wife first. Searching for 'wife nationality' directly is forbidden. I haven't searched for his name yet.",
  "new_search_query": "James Henry Miller wife name",
  "action": "search"
}}

--- Your Turn ---
Format: JSON only. Keys: "thought_process", "action", "new_search_query" OR "final_answer".
"""

CRITIC_PROMPT = """
You are a strict data filter. I will provide a list of retrieved documents.
Your goal is to select the top {k} documents that are MOST relevant and helpful to answer the query: "{query}".

Documents:
{documents_text}

Criteria:
1. Must contain specific entities or facts related to the query.
2. Filter out documents that just share keywords but have the wrong logic (Distractors).

Output format: Return ONLY a JSON list of the INDICES of the selected documents. Example: [0, 4, 12]
"""

EXTRACTOR_PROMPT = """
The context has become too long. Your task is to perform "Information Extraction".
Read the following text and extract ALL key entities, dates, relationships, and facts relevant to the User's original goal.

Guidelines:
1. DO NOT summarize into a story. List facts.
2. KEEP specific names, dates, and numbers exactly as they appear.

Text to process:
{long_context}
"""

# ================= 3. 定义 Agent 类 =================

class AgenticRagSystem_6:
    # 【修改点 2】__init__ 增加 raw_documents 参数，并构建 BM25 索引
    def __init__(self, vectorstore, raw_documents, llm, reranker):
        self.vectorstore = vectorstore
        self.llm = llm
        self.reranker = reranker
        self.context_memory = []
        self.max_iterations = 9
        self.search_history = set()

        # 初始化 BM25 (内存中构建倒排索引)
        print("   ⚙️ Building BM25 Index for Keyword Search...")
        self.bm25_retriever = BM25Retriever.from_documents(raw_documents)
        self.bm25_retriever.k = 30 # 默认 BM25 召回数

    def format_docs(self, docs):
        return "\n\n".join([f"[Doc {i}] {d.page_content}" for i, d in enumerate(docs)])

    # 【修改点 3】重写 retrieve_and_rerank，实现 向量+BM25 并行检索与融合
    def retrieve_and_rerank(self, query, top_k_initial=50, top_k_final=20):
        print(f"   🔍 [Hybrid Retrieve] Searching for: '{query}'...")

        # 1. 向量检索 (Vector Search)
        vector_docs = self.vectorstore.similarity_search(query, k=top_k_initial)

        # 2. 关键词检索 (BM25 Search)
        self.bm25_retriever.k = top_k_initial
        keyword_docs = self.bm25_retriever.invoke(query)

        # 3. 混合去重 (Merge & Dedup)
        # 使用 page_content 作为唯一键
        unique_docs_map = {}
        for d in vector_docs + keyword_docs:
            if d.page_content not in unique_docs_map:
                unique_docs_map[d.page_content] = d

        initial_docs = list(unique_docs_map.values())
        print(f"      - Vector: {len(vector_docs)}, BM25: {len(keyword_docs)} -> Merged: {len(initial_docs)}")

        if not initial_docs: return []

        # 4. Rerank (保持原有逻辑，只是输入变成了混合后的 initial_docs)
        pairs = [[query, d.page_content] for d in initial_docs]
        scores = self.reranker.score(pairs)
        sorted_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        top_docs = [initial_docs[i] for i in sorted_indices[:top_k_final]]

        print(f"   📉 [Rerank] Filtered {len(initial_docs)} -> {len(top_docs)} docs.")
        return top_docs

    def llm_critic_filter(self, query, docs, select_k=5):
        if not docs: return []
        print(f"   🧠 [LLM 2 Critic] Batch processing {len(docs)} docs...")
        docs_text = self.format_docs(docs)

        msg = ChatPromptTemplate.from_template(CRITIC_PROMPT).format_messages(
            k=select_k, query=query, documents_text=docs_text
        )
        response = self.llm.invoke(msg).content
        try:
            match = re.search(r'\[.*\]', response, re.DOTALL)
            if match:
                indices = json.loads(match.group())
                selected = [docs[i] for i in indices if i < len(docs)]
                print(f"   ✅ [LLM 2 Critic] Selected {len(selected)} high-quality docs.")
                return selected
        except:
            return docs[:3]
        return docs[:select_k]

    def llm_compressor(self):
        print("   🗜️ [LLM 3 Compressor] Context > 15 docs. Extracting Information...")
        full_text = "\n".join(self.context_memory)
        msg = ChatPromptTemplate.from_template(EXTRACTOR_PROMPT).format_messages(long_context=full_text)
        extracted_facts = self.llm.invoke(msg).content
        self.context_memory = [extracted_facts]
        print("   ✅ Context compressed.")

    def run(self, user_question):
        # 1. 重置状态
        self.context_memory = []
        self.search_history = set()

        print(f"🚀 Start Agentic RAG for: {user_question}")

        iteration = 0
        while iteration < self.max_iterations:
            print(f"\n--- Iteration {iteration + 1} ---")

            context_str = "\n".join(self.context_memory) if self.context_memory else "No external information yet."

            # 2. 构造 Search History 字符串
            if self.search_history:
                history_str = "\n".join([f"- {q}" for q in self.search_history])
            else:
                history_str = "None."

            # 3. Router 决策
            msg = ChatPromptTemplate.from_template(ROUTER_PROMPT).format_messages(
                context_summary=context_str,
                search_history=history_str,
                question=user_question
            )

            router_response = self.llm.invoke(msg).content

            try:
                router_json = json.loads(router_response.replace("```json", "").replace("```", ""))
            except:
                print("Router JSON parsing failed. Ending.")
                break

            thought = router_json.get("thought_process", "No thought provided.")
            action = router_json.get("action")
            print(f"   💭 [Thought]: {thought}")



            if action == "answer":
                print(f"🎉 [LLM 1] Enough Info! Generating Answer.")
                return router_json.get("final_answer")

            elif action == "search":
                new_query = router_json.get("new_search_query")

                # ---------------------------------------------------------
                # 🔥 [De-Dup] 全量历史去重与强制改写
                # (这段逻辑完全保留，未做任何修改)
                # ---------------------------------------------------------
                if new_query in self.search_history:
                    print(f"⚠️ [Loop Detected] LLM repeated query '{new_query}'.")

                    # 强制变策列表
                    strategies = [" marriage", " spouse", " family", " biography", " personal life", " children"]
                    suffix = strategies[iteration % len(strategies)]

                    original_query = new_query
                    new_query = f"{original_query} {suffix}"
                    print(f"🔀 [Force Modify] Changed query to: '{new_query}'")

                # 加入历史
                self.search_history.add(new_query)
                print(f"🤔 [LLM 1] Decided to search: '{new_query}'")

                # 4. 执行检索 (这里调用的是新修改后的 Hybrid 检索方法)
                top_20_docs = self.retrieve_and_rerank(new_query, top_k_initial=50, top_k_final=20)

                # 5. Critic 过滤
                final_docs = self.llm_critic_filter(new_query, top_20_docs, select_k=5)

                if len(final_docs) == 0:
                    print(f"❌ [Failure] Query returned 0 useful docs.")
                    # 不更新 Context，直接下一轮，让 Router 看到这个失败记录
                    iteration += 1
                    continue
                else:
                    print(f"✅ [Success] Added {len(final_docs)} docs to context.")
                    for d in final_docs:
                        self.context_memory.append(d.page_content)

                # 6. 压缩上下文
                if len(self.context_memory) > 15:
                    self.llm_compressor()

            iteration += 1

        return "Max iterations reached. Could not find complete answer."

In [ ]:
agent_6 = AgenticRagSystem_6(vectorstore, docs, llm, reranker)

# 测试一个多跳问题
# 示例：HotpotQA 经典问题
# 假设问题是： "Which magazine was founded first, Arthur's Magazine or First for Women?"
# 这需要先搜 Arthur's Magazine 的时间，再搜 First for Women 的时间，最后比较。

# 我们可以直接用数据集里的问题来测
test_idx = 49
test_q = dataset[test_idx]['question']


final_answer = agent_6.run(test_q)
print(f"Final Answer: {final_answer}")

In [ ]:
# ================= 5. 运行评测循环 (Evaluation Loop) =================

# 准备测试数据
from datasets import Dataset
EVAL_SIZE = 50
test_dataset = dataset.select(range(EVAL_SIZE))

ragas_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

print(f"\n🚀 开始批量评测 (共 {EVAL_SIZE} 条)...")

for i, row in enumerate(tqdm(test_dataset)):
    question = row['question']
    ground_truth = row['answer']

    # [关键步骤] 1. 清空 Agent 的记忆，防止上一题的文档干扰这一题
    agent_6.context_memory = []

    # [关键步骤] 2. 运行 Agent
    try:
        model_answer = agent_6.run(question)
    except Exception as e:
        model_answer = f"Error: {str(e)}"

    # [关键步骤] 3. 获取 Agent 最终使用的上下文 (用于 RAGAS 评分)
    final_contexts = agent_6.context_memory.copy()

    # --- 实时日志打印 (已修改：每一题都打印，包含 Gold Answer) ---
    print(f"\n[Case {i+1}]")
    print(f"🔍 Question:  {question}")
    print(f"🤖 Agent Ans: {model_answer}")
    print(f"🎯 Gold Ans:  {ground_truth}")   # 新增这一行
    print(f"📄 Docs Used: {len(final_contexts)}")
    print("-" * 50)

    # --- 收集数据 ---
    ragas_data["question"].append(question)
    ragas_data["answer"].append(model_answer)
    ragas_data["contexts"].append(final_contexts)
    ragas_data["ground_truth"].append(ground_truth)

print("\n✅ 所有推理完成。")



In [ ]:
# ================= 6. 生成 RAGAS 数据集 =================
from datasets import Dataset
eval_dataset_agent_6 = Dataset.from_dict(ragas_data)
print(f"Eval Dataset 生成完毕，包含 {len(eval_dataset_agent_6)} 条记录。")

# ==========================================
# 第六步：修复评估流程 (Split Roles)
# ==========================================

from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
    answer_correctness,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# -----------------------------------------------------------
# 关键修复：定义两个不同的 LLM
# 1. 你的 RAG 系统依然使用 GPT-5 mini (刚才已经跑完了，结果存在 eval_dataset 里了)
# 2. 但我们需要一个支持 temperature=0 的“传统模型”来做裁判
# -----------------------------------------------------------

# 定义专门用于评估的“裁判模型” (使用 gpt-4o 或 gpt-3.5-turbo)
# 它们支持 temperature=0，不会报错
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 包装裁判模型
ragas_judge_llm = LangchainLLMWrapper(judge_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# 定义指标列表
metrics = [
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
    answer_correctness,
]

print("正在使用 GPT-4o 作为裁判进行评估 (这不会影响 GPT-5 mini 生成的答案)...")


# 运行评估
results = evaluate(
    dataset=eval_dataset_agent_6,
    metrics=metrics,
    llm=ragas_judge_llm,      # <--- 这里用裁判模型
    embeddings=ragas_embeddings
)

# 展示结果
print("\n========== RAGAS 评估结果 ==========")
print(results)

# 转换为 Pandas DataFrame
df_results = results.to_pandas()
# 直接显示所有行，不再用 head() 截断
df_results

In [ ]:
csv_filename = "ragas_evaluation_Version6_results.csv"
df_results.to_csv(csv_filename, index=False, encoding='utf-8-sig')

# Re-evaluate

In [ ]:
# ==========================================
# 前置准备：配置 Google API (强制重置版)
# ==========================================
import os
import getpass

# 1. 强制清理旧的 Key (这行代码是新增的)
# 这样无论之前有没有设置过，都会把旧的删掉，触发下面的重新输入
if "GOOGLE_API_KEY" in os.environ:
    del os.environ["GOOGLE_API_KEY"]
    print("已清除旧的 API Key，请重新输入。")

# 2. 重新输入配置
if "GOOGLE_API_KEY" not in os.environ:
    # 注意：输入时密码不会显示在屏幕上，输完按回车即可
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("请输入新的 Google API Key: ")

print("API Key 配置更新完毕！")

# ==========================================
# 后续代码保持不变...
# ==========================================

In [ ]:
# ==========================================
# 第六步：执行评估 (Tier 1 高速并行模式)
# ==========================================

from langchain_google_genai import ChatGoogleGenerativeAI
from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
    answer_correctness,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# -----------------------------------------------------------
# 裁判模型配置 (Gemini 2.0 Flash)
# -----------------------------------------------------------
judge_llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,          # 保持 0 以确保作为裁判的评分一致性
    max_output_tokens=8192,
    max_retries=3,          # 付费版比较稳定，常规重试即可
)

ragas_judge_llm = LangchainLLMWrapper(judge_llm)

# 包装 Embeddings
try:
    ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)
except NameError:
    print("⚠️ 未找到 embeddings 变量，请确保你已经初始化了 Embedding 模型。")

# 定义指标
metrics = [
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
    answer_correctness,
]

# 注入裁判模型
for metric in metrics:
    metric.llm = ragas_judge_llm

# -----------------------------------------------------------
# 运行配置 (已解锁 Tier 1 性能)
# -----------------------------------------------------------
# 关键修改：利用付费版的高 Rate Limit 进行并行加速
my_run_config = RunConfig(
    max_workers=16,  # 【核心修改】从 1 改为 16，大幅提升评估速度
    timeout=600,     # 单个请求超时时间
    max_retries=3    # 降低重试阈值
)

print(f"正在使用 {judge_llm.model} (Tier 1 Mode) 进行高速并行评估...")

# 运行评估
results = evaluate(
    dataset=eval_dataset_agent_6,
    metrics=metrics,
    llm=ragas_judge_llm,
    embeddings=ragas_embeddings,
    raise_exceptions=False,
    run_config=my_run_config
)

# 展示结果
print("\n========== RAGAS 评估结果 (Gemini 2.0 Flash) ==========")
df_results = results.to_pandas()

# 设置显示选项
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

df_results

## 1 first Agentic Search

In [ ]:
import json
from typing import Annotated, List, Literal, Dict
from typing_extensions import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

from langgraph.graph import END, StateGraph, START

# ==========================================
# 1. 定义状态 (State)
# ==========================================
class GraphState(TypedDict):
    """
    定义图的状态，所有节点共享这些数据
    """
    question: str                # 用户原始问题
    generation: str              # LLM 生成的最终答案
    documents: List[Document]    # 当前检索到的文档列表
    retry_count: int             # 重试计数器 (防止死循环)

# ==========================================
# 2. 初始化模型 (Model Routing)
# ==========================================
# 大脑：负责生成最终答案、重写查询 (使用强模型)
llm_reasoner = ChatOpenAI(model="gpt-5-mini", temperature=0)

# 判官：负责文档评分 (使用便宜的小模型，省钱关键！)
llm_critic = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ==========================================
# 3. 定义组件：批处理评分器 (Batch Grader)
# ==========================================

# 定义结构化输出，强迫 LLM 返回由 Yes/No 组成的列表
class BatchGrade(BaseModel):
    """Binary scores for check relevance of multiple documents."""
    # 对应文档列表的评分结果，顺序必须一致
    scores: List[str] = Field(
        description="List of 'yes' or 'no' scores corresponding to the order of the documents."
    )

# 使用 structured_output 强制输出 JSON
structured_llm_grader = llm_critic.with_structured_output(BatchGrade)

# 批处理 Prompt
system_prompt_grader = """You are a grader assessing relevance of retrieved documents to a user question.
You will receive a list of documents.
For EACH document, determine if it contains keyword(s) or semantic meaning related to the user question.
Return a list of 'yes' or 'no' strings. The length of the list must match the number of documents.
Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""

grader_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt_grader),
        ("human", "User question: {question}\n\nDocuments:\n{documents_str}"),
    ]
)

batch_grader_chain = grader_prompt | structured_llm_grader

# ==========================================
# 4. 定义图节点 (Nodes)
# ==========================================

def retrieve(state):
    """
    检索节点：从向量库获取文档
    """
    print("---RETRIEVE---")
    question = state["question"]

    # 使用你之前定义的 retriever (Top-K)
    # 注意：这里假设全局变量 'retriever' 已经存在 (来自于上一段代码的 Chroma)
    documents = retriever.invoke(question)

    return {"documents": documents, "question": question}

def grade_documents_batch(state):
    """
    批处理评分节点：一次性评估所有文档
    """
    print("---CHECK RELEVANCE (BATCH)---")
    question = state["question"]
    documents = state["documents"]

    # 1. 准备输入：将文档拼接成带序号的字符串，方便 LLM 识别
    doc_txt_list = [f"[Doc {i}] {doc.page_content}" for i, doc in enumerate(documents)]
    documents_str = "\n\n".join(doc_txt_list)

    # 2. 调用 LLM (Batch Call) - 这里只消耗 1 次 API 调用！
    grade_result = batch_grader_chain.invoke({
        "question": question,
        "documents_str": documents_str
    })

    # 3. 解析结果并过滤
    filtered_docs = []
    scores = grade_result.scores

    # 安全检查：防止 LLM 发疯输出长度不一致
    safe_len = min(len(scores), len(documents))

    for i in range(safe_len):
        score = scores[i].lower()
        if score == "yes":
            print(f"  - Doc {i}: RELEVANT")
            filtered_docs.append(documents[i])
        else:
            print(f"  - Doc {i}: IRRELEVANT (Filtered)")

    return {"documents": filtered_docs, "question": question}

def generate(state):
    """
    生成节点：基于过滤后的文档回答
    """
    print("---GENERATE---")
    question = state["question"]
    documents = state["documents"]

    # 简单的 RAG 生成链
    prompt = ChatPromptTemplate.from_template(
        """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

        Question: {question}

        Context: {context}

        Answer:"""
    )

    # 格式化文档
    context_str = "\n\n".join([d.page_content for d in documents])

    rag_chain = prompt | llm_reasoner | StrOutputParser()
    generation = rag_chain.invoke({"context": context_str, "question": question})

    return {"generation": generation}

def transform_query(state):
    """
    重写查询节点：当找不到相关文档时触发
    """
    print("---TRANSFORM QUERY---")
    question = state["question"]
    documents = state["documents"]
    retry_count = state.get("retry_count", 0)

    # 简单的重写 Prompt
    msg = [
        HumanMessage(content=f"""Look at the input and try to reason about the underlying semantic intent / meaning.
        Here is the initial question:
        {question}

        Formulate an improved question provided the initial question failed to retrieve relevant documents.
        Only return the improved question string.""")
    ]

    better_question = llm_reasoner.invoke(msg).content
    print(f"  - Old Query: {question}")
    print(f"  - New Query: {better_question}")

    return {"question": better_question, "retry_count": retry_count + 1}

# ==========================================
# 5. 定义条件边 (Edges)
# ==========================================

def decide_to_generate(state):
    """
    决策逻辑：有相关文档 -> 生成；无相关文档 -> 重试
    """
    filtered_documents = state["documents"]
    retry_count = state.get("retry_count", 0)

    if not filtered_documents:
        # 如果过滤后一个文档都没剩下
        if retry_count >= 3: # 避免死循环，最多重试 3 次
            print("---DECISION: MAX RETRIES REACHED, STOP---")
            return "give_up" # 或者强行生成

        print("---DECISION: ALL DOCUMENTS IRRELEVANT, TRANSFORM QUERY---")
        return "transform_query"
    else:
        # 有相关文档
        print("---DECISION: GENERATE---")
        return "generate"

# ==========================================
# 6. 构建图 (Graph Construction)
# ==========================================

workflow = StateGraph(GraphState)

# 添加节点
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents_batch) # <--- 核心优化点
workflow.add_node("generate", generate)
workflow.add_node("transform_query", transform_query)

# 构建流程
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "grade_documents")

# 添加条件分支
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "transform_query": "transform_query",
        "generate": "generate",
        "give_up": END # 可以在这里加一个 fallback 节点
    },
)

# 闭环：重写后重新检索
workflow.add_edge("transform_query", "retrieve")
workflow.add_edge("generate", END)

# 编译图
app = workflow.compile()

print("✅ Agentic RAG Graph (with Batch Grader) 已构建完成！")

# ==========================================
# 7. 运行测试
# ==========================================
# 随便测一个之前失败的或者难的 Case
inputs = {"question": " What nationality was James Henry Miller's wife?", "retry_count": 0}

print("\n🚀 开始运行 Agent...")
for output in app.stream(inputs):
    for key, value in output.items():
        # 这里只打印节点名称，详细日志在节点内部已经打印了
        # print(f"Finished Node: {key}")
        pass

print("\n🏁 最终结果：")
# 这里的 output 是最后一次 yield 的结果，通常包含 'generation'
# 由于 stream 的机制，我们需要捕获最后的状态
# 也可以用 app.invoke(inputs) 直接拿结果
final_result = app.invoke(inputs)
print(final_result["generation"])

In [ ]:
from datasets import Dataset
from tqdm import tqdm
import pandas as pd

# ================= 配置区 =================
# Agent 比较慢，建议先跑 10-20 条看个趋势，不要一次跑 50 条
AGENT_EVAL_SIZE = 50
test_dataset = dataset.select(range(AGENT_EVAL_SIZE))

print(f"🚀 开始评估 Agentic RAG (共 {AGENT_EVAL_SIZE} 条)...")
print("-" * 50)

# 准备数据容器
agent_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": [],
    "trace": [] # 额外记录一下：Agent 到底有没有触发重写 (用于分析)
}

# ================= 1. 批量推理 (Agent Running) =================
for i, row in enumerate(tqdm(test_dataset)):
    question = row['question']
    ground_truth = row['answer']

    # 构造 Graph 输入
    inputs = {"question": question, "retry_count": 0}

    # --- 调用 Agent ---
    # app 是你在上一段代码编译好的 LangGraph 对象
    try:
        final_state = app.invoke(inputs)

        # 提取结果
        answer = final_state.get("generation", "No Answer Generated")

        # 提取 Agent 最终选用的文档
        # 注意：如果触发了重写，这里的文档是第二轮检索的结果
        final_docs = final_state.get("documents", [])
        contexts = [doc.page_content for doc in final_docs]

        # 记录是否触发了重写 (通过 retry_count 判断)
        retry_cnt = final_state.get("retry_count", 0)
        trace_info = "Direct" if retry_cnt == 0 else f"Rewritten ({retry_cnt} times)"

    except Exception as e:
        print(f"\n❌ Error on index {i}: {e}")
        answer = "Error"
        contexts = []
        trace_info = "Error"

    # --- 实时打印 (方便你观察 Agent 是否变聪明了) ---
    print(f"\n[第 {i+1} 题] [{trace_info}]")
    print(f"Q: {question}")
    print(f"A: {answer}")
    print("-" * 30)

    # 收集数据
    agent_data["question"].append(question)
    agent_data["answer"].append(answer)
    agent_data["contexts"].append(contexts)
    agent_data["ground_truth"].append(ground_truth)
    agent_data["trace"].append(trace_info)

# 转换为 Dataset
agent_eval_dataset = Dataset.from_dict(agent_data)
print("\n✅ Agent 推理完成，准备进行 RAGAS 打分...")



In [ ]:
# ================= 2. RAGAS 评分 (Evaluation) =================

from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# 依然使用 gpt-4o-mini 做裁判，省钱且够用
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
ragas_judge_llm = LangchainLLMWrapper(judge_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

metrics = [
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
]

print("正在启动裁判模型打分...")

agent_results = evaluate(
    dataset=agent_eval_dataset,
    metrics=metrics,
    llm=ragas_judge_llm,
    embeddings=ragas_embeddings
)

# ================= 3. 展示结果与对比 =================
print("\n========== Agentic RAG 评估结果 ==========")
print(agent_results)

# 转为 DataFrame 展示
df_agent = agent_results.to_pandas()

# 把 Trace 列加进去，方便分析哪些题触发了重写
df_agent["trace"] = agent_data["trace"]

# 看看触发了重写的题目表现如何
print("\n--- 复杂查询(触发重写)的样本 ---")
print(df_agent[df_agent['trace'].str.contains("Rewritten")][['user_input', 'context_recall', 'trace']])

# 保存结果
df_agent.to_csv("agent_rag_evaluation_results.csv", index=False, encoding='utf-8-sig')

In [ ]:
import json
from typing import Annotated, List, Literal, Dict
from typing_extensions import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

from langgraph.graph import END, StateGraph, START

# ==========================================
# 0. 核心配置修改：重置 Retriever 为 Top-50
# ==========================================
# 假设 vectorstore 已经存在 (来自于之前的 Chroma)
# 我们在这里强制覆盖它的搜索参数，让它一次吐出 50 条
retriever_top_50 = vectorstore.as_retriever(search_kwargs={"k": 50})

print("✅ Retriever 已配置为获取 Top-50 文档。")

# ==========================================
# 1. 定义状态 (State)
# ==========================================
class GraphState(TypedDict):
    question: str
    generation: str
    documents: List[Document]
    retry_count: int
    gen_retry_count: int

# ==========================================
# 2. 初始化模型
# ==========================================
# 大脑：生成和推理用强模型
llm_reasoner = ChatOpenAI(model="gpt-5-mini")
# 判官：过滤和检查用快模型 (处理 50 个文档的长 Context 毫无压力)
llm_critic = ChatOpenAI(model="gpt-5-mini")

# ==========================================
# 3. 组件：Batch Grader (支持长列表)
# ==========================================
class BatchGrade(BaseModel):
    """Binary scores for check relevance of multiple documents."""
    # 这里要求 LLM 返回的列表长度必须和输入文档数一致
    scores: List[str] = Field(
        description="List of 'yes' or 'no' scores. The list length MUST match the number of input documents."
    )

structured_llm_grader = llm_critic.with_structured_output(BatchGrade)

# 优化 Prompt，让它能够稳定处理大量文档
system_prompt_grader = """You are a strict grader assessing relevance of retrieved documents to a user question.
You will receive a list of up to 50 documents.
For EACH document, determine if it contains keyword(s) or semantic meaning related to the user question.

IMPORTANT:
1. Be strict. If a document is just a distractor (e.g., same name but wrong entity), mark it 'no'.
2. Return a list of 'yes' or 'no' strings.
3. The length of your output list MUST match the number of input documents exactly.
"""

grader_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt_grader),
        ("human", "User question: {question}\n\nDocuments:\n{documents_str}"),
    ]
)

batch_grader_chain = grader_prompt | structured_llm_grader

# ==========================================
# 4. 定义节点 (Nodes)
# ==========================================

def retrieve_top_50(state):
    """
    检索节点：一次性拉取 50 个文档
    """
    print("---RETRIEVE (TOP 50)---")
    question = state["question"]

    # 调用 Top-50 Retriever
    documents = retriever_top_50.invoke(question)
    print(f"  - Retrieved {len(documents)} documents from vector store.")

    return {"documents": documents, "question": question}

def grade_documents_batch(state):
    """
    过滤节点：从 50 个里挑出有用的
    """
    print("---CRITIC FILTERING (BATCH)---")
    question = state["question"]
    documents = state["documents"]

    # 构造长 Context 输入
    # 注意：50 个 chunk 可能会有 5k-10k token，gpt-4o-mini 吃得消
    doc_txt_list = [f"[Doc {i}] {doc.page_content}" for i, doc in enumerate(documents)]
    documents_str = "\n\n".join(doc_txt_list)

    try:
        # 调用 LLM 进行批量判决
        grade_result = batch_grader_chain.invoke({
            "question": question,
            "documents_str": documents_str
        })
        scores = grade_result.scores
    except Exception as e:
        print(f"  - Error in grading: {e}")
        # 如果评分崩了，为安全起见，保留前 5 个，避免 pipeline 断裂
        scores = ["yes"] * 5 + ["no"] * (len(documents) - 5)

    filtered_docs = []

    # 对齐检查
    safe_len = min(len(scores), len(documents))

    relevant_count = 0
    for i in range(safe_len):
        if scores[i].lower() == "yes":
            filtered_docs.append(documents[i])
            relevant_count += 1

    print(f"  - Filtered: Kept {relevant_count} relevant docs out of {len(documents)}.")

    # 如果 50 个里一个能打的都没有，保留空列表，后面会触发 transform_query
    return {"documents": filtered_docs, "question": question}

def generate(state):
    """
    生成节点：只用过滤后的高质量文档
    """
    print("---GENERATE---")
    question = state["question"]
    documents = state["documents"]
    gen_retry_count = state.get("gen_retry_count", 0)

    # 如果过滤后文档太多（比如 20 个），可能会撑爆 context 或让 generator 变糊涂
    # 这里可以做一个截断，比如只取前 10 个最相关的（假设 Critic 的顺序有意义，或者这里就是保留全部）
    # 为了 HotpotQA，我们保留全部相关文档

    context_str = "\n\n".join([d.page_content for d in documents])

    prompt = ChatPromptTemplate.from_template(
        """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question.
        If you don't know the answer, just say that you don't know.

        Question: {question}

        Context:
        {context}

        Answer:"""
    )

    rag_chain = prompt | llm_reasoner | StrOutputParser()
    generation = rag_chain.invoke({"context": context_str, "question": question})

    return {"generation": generation, "gen_retry_count": gen_retry_count + 1}

def transform_query(state):
    print("---TRANSFORM QUERY---")
    question = state["question"]
    retry_count = state.get("retry_count", 0)

    msg = [HumanMessage(content=f"Look at the input and try to reason about the underlying semantic intent. Initial question: {question}. Return only the improved question string.")]
    better_question = llm_reasoner.invoke(msg).content
    print(f"  - New Query: {better_question}")

    return {"question": better_question, "retry_count": retry_count + 1}

# ==========================================
# 5. 定义边 (Edges)
# ==========================================

def decide_to_generate(state):
    filtered_documents = state["documents"]
    retry_count = state.get("retry_count", 0)

    if not filtered_documents:
        # 50 个文档全军覆没
        if retry_count >= 3:
            print("---DECISION: STOP (MAX RETRIES)---")
            return "give_up"
        print("---DECISION: NO RELEVANT DOCS found in Top 50, TRANSFORM QUERY---")
        return "transform_query"
    else:
        print("---DECISION: GENERATE---")
        return "generate"

# (此处省略了之前添加的 hallucination_grader，如果你需要可以加回来，逻辑是一样的)
# 为了聚焦你 "Retrieve 50 -> Critic -> Generate" 的核心诉求，我们先简化结尾

# ==========================================
# 6. 构建图
# ==========================================

workflow = StateGraph(GraphState)

# Nodes
workflow.add_node("retrieve", retrieve_top_50)      # <-- 改用 Top 50
workflow.add_node("grade_documents", grade_documents_batch)
workflow.add_node("generate", generate)
workflow.add_node("transform_query", transform_query)

# Edges
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "grade_documents") # 50个文档流入 Critic

workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "transform_query": "transform_query", # 全被过滤 -> 重写
        "generate": "generate",               # 有保留 -> 生成
        "give_up": END
    },
)

workflow.add_edge("transform_query", "retrieve")
workflow.add_edge("generate", END)

app_top50 = workflow.compile()
print("✅ Agent (Top-50 Retrieval Version) 构建完成！")

# ==========================================
# 7. 测试
# ==========================================
inputs = {"question": "The Oberoi family is part of a hotel company that has a head office in what city?", "retry_count": 0, "gen_retry_count": 0}

print("\n🚀 运行 Top-50 Agent...")
for output in app_top50.stream(inputs):
    pass

final = app_top50.invoke(inputs)
print("\n🏁 Final Answer:", final["generation"])

In [ ]:
from datasets import Dataset
from tqdm import tqdm
import pandas as pd
from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# ================= 配置区 =================
# 建议先测 20-50 条。因为现在的流程比较长 (检索50条 -> 评分 -> 生成 -> 幻觉检查)，速度会慢一些。
AGENT_EVAL_SIZE = 50
test_dataset = dataset.select(range(AGENT_EVAL_SIZE))

print(f"🚀 开始评估 Agentic RAG (Top-50 + Critic + Hallucination Check)...")
print(f"样本量: {AGENT_EVAL_SIZE}")
print("-" * 50)

# 准备数据容器
agent_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": [],
    "trace_search": [], # 记录检索重试 (Query Rewrite)
    "trace_gen": []     # 记录生成重试 (Hallucination Fix)
}

# ================= 1. 批量推理 (Agent Running) =================
# 确保使用你刚才编译的那个图对象，如果是 app_top50 就用 app_top50
target_graph = app_top50

for i, row in enumerate(tqdm(test_dataset)):
    question = row['question']
    ground_truth = row['answer']

    # 构造 Graph 输入 (注意：现在需要初始化 gen_retry_count)
    inputs = {
        "question": question,
        "retry_count": 0,
        "gen_retry_count": 0
    }

    answer = "Error"
    contexts = []
    trace_search = "Direct"
    trace_gen = "Direct"

    try:
        # 调用 Graph
        final_state = target_graph.invoke(inputs)

        # 1. 提取答案
        answer = final_state.get("generation", "No Answer Generated")

        # 2. 提取最终使用的文档 (Critic 过滤后的)
        final_docs = final_state.get("documents", [])
        contexts = [doc.page_content for doc in final_docs]

        # 3. 分析 Trace (检索重试)
        rc = final_state.get("retry_count", 0)
        trace_search = "Direct" if rc == 0 else f"Rewritten ({rc})"

        # 4. 分析 Trace (生成重试 - 幻觉/质量检查)
        grc = final_state.get("gen_retry_count", 0)
        # 如果 gen_retry_count > 1，说明触发了幻觉重写 (因为第一次生成算 1 次，重试才是 >1)
        # 但我们在代码里逻辑是每次 enter generate node +1。
        # 如果直接通过，它是 1。如果被打回一次，它是 2。
        trace_gen = "Pass" if grc <= 1 else f"Refined ({grc-1} times)"

    except Exception as e:
        print(f"\n❌ Error on index {i}: {e}")

    # --- 实时打印 (Debug 窗口) ---
    # 只打印那些触发了“重试”的有趣样本，避免刷屏
    if trace_search != "Direct" or trace_gen != "Pass":
        print(f"\n[第 {i+1} 题] 🔄 触发修正! Search: {trace_search} | Gen: {trace_gen}")
        print(f"Q: {question}")
        print(f"A: {answer}")
        print("-" * 30)

    # 收集数据
    agent_data["question"].append(question)
    agent_data["answer"].append(answer)
    agent_data["contexts"].append(contexts)
    agent_data["ground_truth"].append(ground_truth)
    agent_data["trace_search"].append(trace_search)
    agent_data["trace_gen"].append(trace_gen)

# 转换为 Dataset
agent_eval_dataset = Dataset.from_dict(agent_data)
print("\n✅ Agent 推理完成。")

# ================= 2. RAGAS 评分 (Evaluation) =================

print("正在启动 RAGAS 裁判模型 (GPT-4o) ...")

# 定义裁判 (使用 gpt-4o 保证评估质量，temperature=0)
judge_llm = ChatOpenAI(model="gpt-4o", temperature=0)
ragas_judge_llm = LangchainLLMWrapper(judge_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

metrics = [
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
]

agent_results = evaluate(
    dataset=agent_eval_dataset,
    metrics=metrics,
    llm=ragas_judge_llm,
    embeddings=ragas_embeddings
)

# ================= 3. 结果展示 =================
print("\n========== Agentic RAG (Top-50) 评估结果 ==========")
print(agent_results)

# 转为 DataFrame
df_agent = agent_results.to_pandas()

# 把 Trace 列拼回去，方便你分析
df_agent["trace_search"] = agent_data["trace_search"]
df_agent["trace_gen"] = agent_data["trace_gen"]

# 💡 分析亮点：看看那些触发了重写的题目，最后得分高不高？
print("\n--- 🕵️‍♀️ 深度分析: 触发重写的样本表现 ---")
rewritten_df = df_agent[
    (df_agent['trace_search'].str.contains("Rewritten")) |
    (df_agent['trace_gen'].str.contains("Refined"))
]

if not rewritten_df.empty:
    print(rewritten_df[['user_input', 'context_recall', 'faithfulness', 'trace_search', 'trace_gen']])
else:
    print("本次测试中没有触发重写的样本 (可能是 Top-50 召回太强了，一次就搞定了)。")

# 保存
df_agent.to_csv("agent_top50_eval_results.csv", index=False, encoding='utf-8-sig')
print("\n✅ 结果已保存至 agent_top50_eval_results.csv")